# forgeformer

© Aritro 'sortira' Shome

a transformer* which adds 2 two-digit numbers, and returns the result^.

link to the blog is [here]() and link to the video is [here]()

`* - not all parts of the transformer are present. only attention layers and select few components are used. there are some hacks used in the unembedding and embedding portions to help me focus on the attention head and its function portion.`

`^ - result has to be a 2 digit number as well. hundredths position supported only because of the "hack" implemented in the unembedding portion`

In [3]:
import numpy as np

In [4]:
d_model = 3
h = 1
d_k = d_model // h

wq1 = np.array([[0,0,0], [0,0,0], [0,0, 10]]) 
wk1 = np.array([[0,0,0], [0,0,0], [0,0,-10]]) 
wv1 = np.array([[0,0,0], [2.0,0,0], [0,0,0]])

wq2 = np.array([[0,0,0], [0,0,0], [0,0, 10]])
wk2 = np.array([[0,0,0], [0,0,0], [0,0, 10]]) 
wv2 = np.array([[0,0,0], [0,3.0,0], [0,0,0]])

def softmax(matrix, axis = 1):
    x_max = np.max(matrix, axis=axis, keepdims=True)
    e_x = np.exp(matrix - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

In [ ]:
def forgeformer(tokens: list[int]):
    n = len(tokens)
    x = np.zeros((n, d_model), dtype = np.float64)
    states_map = {}

    # converting token to embeddings [this portion is a hack]
    for i in range(n):
        if i == 4: # EOS token
            x[i, 1] = 0.0 # EOS must not carry digit values into the sum
            x[i, 2] = 1.0 
        else:
            x[i, 1] = tokens[i]
            x[i, 2] = 1.0 if i % 2 == 0 else -1.0 
            
    # --- Layer 1: Units Place---
    states_map["x1"] = x
    states_map["k1"] = x @ wk1
    states_map["q1"] = x @ wq1
    states_map["v1"] = x @ wv1
    states_map["kq1"] = (x @ wq1) @ (x @ wk1).T
    attn1 = softmax((x @ wq1) @ (x @ wk1).T)
    states_map["attn1"] = attn1
    x = x + (attn1 @ (x @ wv1))
    states_map["x2"] = x
    
    # --- Layer 2: Tens Place---
    states_map["k2"] = x @ wk2
    states_map["q2"] = x @ wq2
    states_map["v2"] = x @ wv2
    states_map["kq2"] = (x @ wq2) @ (x @ wk2).T
    attn2 = softmax((x @ wq2) @ (x @ wk2).T ) 
    states_map["attn2"] = attn2
    x = x + (attn2 @ (x @ wv2))
    states_map["output"] = x

    # --- Readout with Manual Carry [this portion is a hack] ---
    units_sum = x[-1, 0]
    tens_sum = x[-1, 1]
    
    carry = 0
    if units_sum >= 10:
        carry = 1
        units_sum -= 10
        
    return (int((tens_sum + carry) * 10 + units_sum), states_map)

In [ ]:
x = int(input('Enter first atmost 2 digit number'))
y = int(input('Enter second atmost 2 digit number'))
ans = None
states = None
if x > 99 or y > 99 or x < 0 or y < 0:
    print('Invalid input')
else:
    ans, states = forgeformer([x // 10, x % 10, y // 10, y % 10, -1])
    print(f"{x} + {y} = {ans}")

In [ ]:
states

for queries/doubts/errata, reach out to me at

[Twitter](https://x.com/silicognition)

[Personal site](https://aritro.is-a.dev/)

[Email](mailto:aritro.shome.official@gmail.com)